# 01 — Retail Transaction Data Exploration

**Project:** Transaction Anomaly Detection  
**Phase:** 01 — Exploratory Data Analysis (EDA)  
**Source:** UCI Online Retail transaction-line dataset (2010–2011)

### Business question
At the time an order is submitted, which **eligible retail purchases** appear sufficiently unusual to warrant manual investigation?

This notebook describes the available evidence and data-quality constraints. It **does not** train a model or claim to identify confirmed fraud. Anomalies are not fraud labels.

### Decisions already agreed
- Score **orders/invoices**, not individual item rows, using the complete basket as known at submission.
- Include eligible purchases even when `CustomerID` is missing.
- Keep zero-priced items inside otherwise chargeable baskets.
- Exclude cancellation and non-purchase administrative records from the candidate *purchase-scoring* population, while preserving them for data audit.
- Later test customer-history windows of **24 hours, 30 days, 90 days**, plus all prior history. None are implemented in this notebook.

**Research questions:** What is an order in this export? What information is missing? What transactions are exceptional? Which records require explicit eligibility rules before modeling?

## 1. Environment and dataset location

Put the **public** `online_retail.csv.zip` file inside `data/` in the project root. The ZIP should contain a single `online_retail.csv` file. The `data/` directory is deliberately ignored by Git; do not commit the dataset.

Run the notebook from either the repository root or the `notebooks/` directory. No external downloads or credentials are required.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

filenames = ("online_retail.csv.zip", "online_retail.csv(1).zip", "online_retail.csv")
search_roots = (Path.cwd(), Path.cwd().parent)
candidates = [root / "data" / filename for root in search_roots for filename in filenames]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Place online_retail.csv.zip in the repository's data/ folder, then rerun. "
        f"Searched: {[str(path) for path in candidates]}"
    )

print(f"Dataset: {data_path.resolve()}")
print(f"Pandas: {pd.__version__}")

## 2. Load and validate the raw data

Each row is a **line item**, so repeated `InvoiceNo` values are expected. The exported `index` field is only a row identifier; it is not a predictor. Preserve the raw data during EDA.

In [ ]:
raw = pd.read_csv(
    data_path,
    dtype={"InvoiceNo": "string", "StockCode": "string", "Country": "string"},
    low_memory=False,
)
required = {
    "InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate",
    "UnitPrice", "CustomerID", "Country",
}
missing_columns = required.difference(raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = raw.copy()
df["event_at"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M", errors="coerce")

print(f"Rows: {len(df):,} | Columns in original CSV: {raw.shape[1]}")
print(f"Invoice IDs: {df['InvoiceNo'].nunique():,}")
print(f"Parsed timestamps: {df['event_at'].notna().sum():,} / {len(df):,}")
print(f"Observed period: {df['event_at'].min()} to {df['event_at'].max()}")
display(df.head(5))
display(raw.dtypes.rename("dtype").to_frame())

## 3. Completeness and basic value checks

An absent customer identifier must be treated as **unknown history**, not as evidence of risk. Similarly, zero or negative quantity and price values may represent returns, bookkeeping, or adjustments; examine them before deciding what to exclude.

In [ ]:
missing = pd.DataFrame({
    "missing_rows": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
}).sort_values("missing_pct", ascending=False)
display(missing)

quality_counts = pd.Series({
    "unparseable_timestamps": int(df["event_at"].isna().sum()),
    "missing_invoice_id": int(df["InvoiceNo"].isna().sum()),
    "negative_quantity_lines": int((df["Quantity"] < 0).sum()),
    "zero_quantity_lines": int((df["Quantity"] == 0).sum()),
    "zero_unit_price_lines": int((df["UnitPrice"] == 0).sum()),
    "negative_unit_price_lines": int((df["UnitPrice"] < 0).sum()),
    "distinct_countries": int(df["Country"].nunique(dropna=True)),
    "identified_customers": int(df["CustomerID"].nunique(dropna=True)),
}, name="count")
display(quality_counts.to_frame())

## 4. Distinguish purchase lines, cancellations, and adjustments

For now these are **EDA flags**, not a permanent cleaning function.

- Invoice IDs beginning with `C` appear to represent cancellations.
- Positive-quantity, positive-price lines are chargeable purchase lines.
- Positive-quantity, zero-price lines are preserved as free items *when part of a paid order*.
- `StockCode == 'B'` with a bad-debt description indicates administrative adjustments.

Retain the original rows, including cancellations, for investigation and future outcome analysis. **Do not use a future cancellation as an input to an earlier purchase score.**

In [ ]:
df["cancel_prefix"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)
df["chargeable_line"] = (df["Quantity"] > 0) & (df["UnitPrice"] > 0)
df["free_item_line"] = (df["Quantity"] > 0) & (df["UnitPrice"] == 0)
df["negative_qty_line"] = df["Quantity"] < 0
df["bad_debt_line"] = (
    df["StockCode"].str.upper().eq("B")
    & df["Description"].str.contains("bad debt", case=False, na=False)
).fillna(False)
df["positive_purchase_amount"] = (
    df["Quantity"] * df["UnitPrice"]
).where(df["chargeable_line"], 0.0)
df["positive_units"] = df["Quantity"].where(df["Quantity"] > 0, 0)
df["missing_customer"] = df["CustomerID"].isna()

line_flags = pd.Series({
    "cancellation_prefix_lines": int(df["cancel_prefix"].sum()),
    "chargeable_lines": int(df["chargeable_line"].sum()),
    "positive_quantity_free_lines": int(df["free_item_line"].sum()),
    "bad_debt_adjustment_lines": int(df["bad_debt_line"].sum()),
}, name="count")
display(line_flags.to_frame())
display(df.loc[df["bad_debt_line"], list(raw.columns)])

## 5. Build an invoice-level **audit view**

This is a temporary analytical aggregation, **not** the future training/serving pipeline. We use the latest timestamp among an invoice's lines as a proxy for when the *whole basket* is available. A real checkout event timestamp is unavailable in this dataset.

We intentionally preserve line counts, free items, and warnings about inconsistent timestamps, customer IDs, or countries.

In [ ]:
invoice = (
    df.groupby("InvoiceNo", dropna=False, sort=False)
      .agg(
          first_line_at=("event_at", "min"),
          basket_available_at=("event_at", "max"),
          distinct_timestamps=("event_at", "nunique"),
          n_lines=("StockCode", "size"),
          n_products=("StockCode", "nunique"),
          customer_id=("CustomerID", "first"),
          n_customer_ids=("CustomerID", "nunique"),
          n_countries=("Country", "nunique"),
          n_chargeable_lines=("chargeable_line", "sum"),
          n_free_lines=("free_item_line", "sum"),
          n_negative_qty_lines=("negative_qty_line", "sum"),
          n_bad_debt_lines=("bad_debt_line", "sum"),
          gross_positive_units=("positive_units", "sum"),
          order_value_gbp=("positive_purchase_amount", "sum"),
          missing_customer_id=("missing_customer", "all"),
          is_cancellation=("cancel_prefix", "max"),
      )
)

invoice["record_type"] = np.select(
    [
        invoice["is_cancellation"],
        invoice["n_bad_debt_lines"] > 0,
        invoice["n_chargeable_lines"] == 0,
    ],
    ["cancellation", "administrative adjustment", "no chargeable purchase lines"],
    default="candidate purchase",
)
invoice["candidate_purchase"] = invoice["record_type"].eq("candidate purchase")
purchases = invoice.loc[invoice["candidate_purchase"]].copy()

print(f"Invoice records: {len(invoice):,} | Candidate purchases: {len(purchases):,}")
display(invoice["record_type"].value_counts().rename_axis("record_type").to_frame("invoices"))
display(purchases.head(5))

## 6. Audit invoice integrity and free-item baskets

These flags should trigger **inspection**, not automatic deletion. Identical-looking rows may correspond to separate units/lines, and multiple timestamps for one invoice may reflect system behavior or data errors.

In [ ]:
repeated_lines = raw.duplicated(subset=[c for c in raw.columns if c != "index"], keep="first")
repeat_examples = raw.loc[
    raw.duplicated(subset=[c for c in raw.columns if c != "index"], keep=False)
]

audit_counts = pd.Series({
    "duplicate-looking lines beyond first": int(repeated_lines.sum()),
    "invoices with multiple timestamps": int((invoice["distinct_timestamps"] > 1).sum()),
    "invoices with multiple identified customers": int((invoice["n_customer_ids"] > 1).sum()),
    "invoices with multiple countries": int((invoice["n_countries"] > 1).sum()),
    "candidate purchases with zero-price lines": int((purchases["n_free_lines"] > 0).sum()),
    "candidate purchases without CustomerID": int(purchases["missing_customer_id"].sum()),
}, name="count")
display(audit_counts.to_frame())
print("Example duplicate-looking rows (keep for investigation):")
display(repeat_examples.head(8))
print("Example invoices with multiple timestamps:")
display(invoice.loc[invoice["distinct_timestamps"] > 1].head(8))

## 7. Size and time distributions (candidate purchases only)

Inspect the distribution *before* deciding whether to transform or clip anything. Extreme orders are valuable research cases; never silently drop them just because they are unusual.

The plotting transformations below are **visualization only**. They are not yet model features.

In [ ]:
summary = purchases[["order_value_gbp", "gross_positive_units", "n_lines", "n_products"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
).T
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log1p(purchases["order_value_gbp"].clip(lower=0)), bins=60)
axes[0].set(title="Log(1 + purchase value)", xlabel="Log(1 + GBP)", ylabel="Invoice count")
axes[1].hist(np.log1p(purchases["gross_positive_units"].clip(lower=0)), bins=60)
axes[1].set(title="Log(1 + positive units)", xlabel="Log(1 + units)", ylabel="Invoice count")
plt.tight_layout()
plt.show()

monthly = purchases.groupby(purchases["basket_available_at"].dt.to_period("M")).size()
ax = monthly.plot(kind="bar", figsize=(12, 4), title="Candidate purchase count by month")
ax.set(xlabel="Submission month (proxy)", ylabel="Invoice count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Customer coverage and purchasing frequency

Customer-level observations here are descriptive only; *no future-looking history features are created*. A customer's first appearance in the exported data does **not** prove they are a new customer.

In [ ]:
identified = purchases.loc[purchases["customer_id"].notna()].copy()
orders_per_customer = identified.groupby("customer_id").size()

print(f"Identified customers among candidate purchases: {orders_per_customer.size:,}")
print(f"Candidate purchases without CustomerID: {purchases['missing_customer_id'].sum():,}")
display(orders_per_customer.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("orders_per_customer"))

gaps = (
    identified.sort_values(["customer_id", "basket_available_at"])
    .groupby("customer_id")["basket_available_at"]
    .diff()
    .dt.total_seconds()
    .div(86400)
    .dropna()
)
print(f"Observed consecutive-purchase gaps: {len(gaps):,}")
if not gaps.empty:
    display(gaps.describe(percentiles=[0.5, 0.9, 0.95]).to_frame("days_between_observed_orders"))

## 9. Review concrete anomalous **candidates**, not "fraud cases"

Orders with extraordinary amounts, quantities, or unusual basket composition should be inspected manually. These are **descriptive extremes**, not model predictions or fraud labels.

In [ ]:
columns_to_review = [
    "basket_available_at", "customer_id", "order_value_gbp",
    "gross_positive_units", "n_lines", "n_products", "n_free_lines",
]
print("Highest-value purchase candidates:")
display(purchases.nlargest(10, "order_value_gbp")[columns_to_review])
print("Largest purchase candidates by quantity:")
display(purchases.nlargest(10, "gross_positive_units")[columns_to_review])
print("Purchase candidates containing free items:")
display(purchases.loc[purchases["n_free_lines"] > 0, columns_to_review].head(10))

case_id = "581483"
if case_id in df["InvoiceNo"].values:
    print(f"Raw lines for illustrative invoice {case_id}:")
    display(df.loc[df["InvoiceNo"] == case_id, list(raw.columns)])

## 10. Conclusions to record before Phase 02

**Things this dataset can support:**
- Invoice-level exploratory analysis and point-in-time transaction metadata.
- Investigation of order values, quantities, basket shape, and identified-customer purchase history.
- A retrospective simulation of an order-submission scoring workflow, subject to our full-basket timing assumption.

**Things this dataset cannot establish:**
- Confirmed fraud labels, fraud probability, fraud precision/recall, payment authorization decisions, device/card/IP signals, or actual business loss avoided.
- Whether a same-timestamp order occurred before another order unless an external ordering record exists.

**Carry-forward data decisions:**
1. Candidate purchase invoices have positive-quantity, positive-price purchase lines and are not cancellations or bad-debt adjustments.
2. Preserve zero-priced items inside candidate purchase baskets.
3. Include candidates without customer IDs and explicitly handle missing history later.
4. Do not drop outliers or duplicate-looking lines without investigation.
5. Use timestamp-aware historical features (24h, 30d, 90d, and cumulative) **only from prior known orders** in the next phase.

### Research notes (fill in after running this notebook)
- Which anomalies appear operationally meaningful?
- Which data-integrity issues need special handling?
- Which proposed features are measurable at submission vs. only after the fact?
- Does the invoice-timestamp proxy require an exclusion/sensitivity study?

**Stop here.** Phase 02 will develop candidate features individually and validate them; this notebook intentionally contains no model training.